<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose

This notebook is the **complete implementation specification** for the corresponding `main.ipynb`.

Its purpose is to make the executable notebook fully reproducible from the algorithm alone. A competent reader should be able to rebuild `main.ipynb` **without guessing, inventing, or choosing any meaningful implementation detail**.

The specification therefore defines the exact execution order, libraries, constants, data structures, helper procedures, matrix operations, numerical branches, failure conditions, validation tolerances, diagnostic figures, output filenames, and final checks used by the implementation.

**Rule:** if a coding decision is required to reproduce `main.ipynb`, that decision must be stated explicitly in this notebook.

# Complete Algorithm Specification

The pseudocode below is the authoritative coding blueprint for `main.ipynb`. It is intentionally explicit: every computational step, stored variable, branch, numerical operation, figure requirement, and validation rule needed to reproduce the executable notebook is stated directly.

```text
ALGORITHM Camera_Calibration

  # 0. IMPORTS AND GLOBAL SETTINGS
  IMPORT Path FROM pathlib
  IMPORT cv2
  IMPORT numpy AS np
  IMPORT matplotlib.pyplot AS plt
  CALL np.set_printoptions(precision=6, suppress=True)
  INTERNAL_CORNERS_X ← 8
  INTERNAL_CORNERS_Y ← 6
  SQUARE_SIZE_M ← 0.03
  MIN_VALID_VIEWS ← 3
  DATA_DIR ← Path("../data/calibration_images")
  OUTPUT_DIR ← Path("../outputs/figures")
  CALL OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
  IMAGE_EXTENSIONS ← {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
  REFINE_CRITERIA ← (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

  REQUIRED_OUTPUTS ← [
      "detected_chessboard_corners.png",
      "homography_estimation_pipeline.png",
      "estimated_camera_poses.png",
      "reprojection_results.png",
      "mean_reprojection_error_by_view.png",
      "reprojection_error_distribution.png"
  ]

  # 1. DISCOVER AND LOAD IMAGES
  IF NOT DATA_DIR.is_dir(): RAISE FileNotFoundError
  calibration_image_paths ← sorted supported files in DATA_DIR
  IF calibration_image_paths is empty: RAISE FileNotFoundError
  readable_calibration_views ← []
  FOR image_path IN calibration_image_paths:
      image ← cv2.imread(str(image_path))
      IF image IS None:
          PRINT skipped message
          CONTINUE
      APPEND {path:image_path, name:image_path.name, image:image}
  IF readable_calibration_views is empty: RAISE RuntimeError

  # 2. DETECT AND REFINE CHESSBOARD CORNERS
  FUNCTION detect_and_refine_chessboard_corners(image):
      gray ← cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      found, corners ← cv2.findChessboardCorners(gray, (INTERNAL_CORNERS_X, INTERNAL_CORNERS_Y), None)
      IF NOT found OR corners IS None: RETURN None
      refined ← cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), REFINE_CRITERIA)
      RETURN refined.reshape(-1, 2)
  views ← []
  FOR readable_view IN readable_calibration_views:
      detected_corner_points ← detect_and_refine_chessboard_corners(readable_view["image"])
      IF detected_corner_points IS None: PRINT skipped message; CONTINUE
      IF detected_corner_points.shape != (INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y, 2): RAISE ValueError
      view ← COPY(readable_view)
      view["im_pts"] ← detected_corner_points
      APPEND view TO views
  IF len(views) < MIN_VALID_VIEWS: RAISE RuntimeError

  # 3. BUILD PLANAR WORLD COORDINATES
  FUNCTION build_chessboard_plane_points(
      corners_x=INTERNAL_CORNERS_X,
      corners_y=INTERNAL_CORNERS_Y,
      square_size=SQUARE_SIZE_M
  ):
      points ← np.zeros((corners_x * corners_y, 2), dtype=np.float64)
      index ← 0
      FOR y_index IN range(corners_y):
          FOR x_index IN range(corners_x):
              points[index,0] ← x_index * square_size
              points[index,1] ← y_index * square_size
              index ← index + 1
      RETURN points
  chessboard_plane_points ← build_chessboard_plane_points()
  FOR view IN views: view["plane_pts"] ← chessboard_plane_points.copy()

  # 4. HOMOGENEOUS COORDINATES AND HARTLEY NORMALIZATION
  FUNCTION convert_to_homogeneous_coordinates(points):
      homogeneous ← np.ones((points.shape[0],3), dtype=float)
      homogeneous[:,:2] ← np.asarray(points, dtype=float)
      RETURN homogeneous
  FUNCTION compute_point_normalization_transform(points):
      points ← np.asarray(points, dtype=float)
      centroid ← np.mean(points, axis=0)
      shifted ← points - centroid
      mean_distance ← np.mean(np.linalg.norm(shifted, axis=1))
      IF mean_distance < 1e-12: RAISE ValueError
      scale ← np.sqrt(2.0) / mean_distance
      RETURN np.array([[scale,0,-scale*centroid[0]],[0,scale,-scale*centroid[1]],[0,0,1]], dtype=float)
  FOR view IN views:
      view["T_image"] ← compute_point_normalization_transform(view["im_pts"])
      view["T_plane"] ← compute_point_normalization_transform(view["plane_pts"])
      view["normalized_image"] ← (view["T_image"] @ convert_to_homogeneous_coordinates(view["im_pts"]).T).T
      view["normalized_plane"] ← (view["T_plane"] @ convert_to_homogeneous_coordinates(view["plane_pts"]).T).T

  # 5. NORMALIZED DLT
  FUNCTION build_homography_dlt_matrix(normalized_plane, normalized_image):
      Q ← np.zeros((2*normalized_image.shape[0], 9), dtype=float)
      FOR i IN range(normalized_image.shape[0]):
          X,Y,_ ← normalized_plane[i]
          u,v,_ ← normalized_image[i]
          Q[2*i] ← [X,Y,1,0,0,0,-u*X,-u*Y,-u]
          Q[2*i+1] ← [0,0,0,X,Y,1,-v*X,-v*Y,-v]
      RETURN Q
  FOR view IN views:
      Q ← build_homography_dlt_matrix(view["normalized_plane"], view["normalized_image"])
      _, sQ, VtQ ← np.linalg.svd(Q)
      h ← VtQ[-1]
      view["Q"] ← Q
      view["h"] ← h
      view["dlt_singular_values"] ← sQ
      view["H_normalized"] ← h.reshape(3,3)

  # 6. DENORMALIZE HOMOGRAPHIES
  FUNCTION denormalize_homography_matrix(H_normalized, T_image, T_plane):
      H ← np.linalg.inv(T_image) @ H_normalized @ T_plane
      IF abs(H[2,2]) < 1e-12: RAISE ValueError
      RETURN H / H[2,2]

  expected_corner_count ← INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y
  FOR view IN views:
      view["H"] ← denormalize_homography_matrix(
          view["H_normalized"],
          view["T_image"],
          view["T_plane"]
      )
      IF view["im_pts"].shape != (expected_corner_count, 2): RAISE ValueError
      IF view["plane_pts"].shape != (expected_corner_count, 2): RAISE ValueError
      IF NOT np.all(np.isfinite(view["H"])): RAISE ValueError

  # 7. ZHANG MULTI-VIEW CONSTRAINTS
  FUNCTION build_zhang_constraint_vector(hi, hj):
      RETURN np.array([
          hi[0]*hj[0],
          hi[0]*hj[1] + hi[1]*hj[0],
          hi[1]*hj[1],
          hi[2]*hj[0] + hi[0]*hj[2],
          hi[2]*hj[1] + hi[1]*hj[2],
          hi[2]*hj[2]
      ], dtype=float)

  FUNCTION build_zhang_constraints_for_view(H):
      h1 ← H[:, 0]
      h2 ← H[:, 1]
      v12 ← build_zhang_constraint_vector(h1, h2)
      v11 ← build_zhang_constraint_vector(h1, h1)
      v22 ← build_zhang_constraint_vector(h2, h2)
      RETURN np.vstack((v12, v11 - v22))

  V ← np.vstack([
      build_zhang_constraints_for_view(view["H"])
      FOR view IN views
  ])
  _, singular_values, Vt ← np.linalg.svd(V)
  b ← Vt[-1]

  # 8. RECOVER INTRINSICS
  b11, b12, b22, b13, b23, b33 ← b
  denominator ← b11*b22 - b12**2
  IF abs(denominator) < 1e-12: RAISE ValueError
  v0 ← (b12*b13 - b11*b23) / denominator
  lambda_ ← b33 - (b13**2 + v0*(b12*b13 - b11*b23)) / b11

  IF lambda_/b11 <= 0 OR lambda_*b11/denominator <= 0:
      b ← -b
      b11, b12, b22, b13, b23, b33 ← b
      denominator ← b11*b22 - b12**2
      v0 ← (b12*b13 - b11*b23) / denominator
      lambda_ ← b33 - (b13**2 + v0*(b12*b13 - b11*b23)) / b11

  alpha ← np.sqrt(lambda_/b11)
  beta ← np.sqrt(lambda_*b11/denominator)
  gamma ← -b12*alpha**2*beta/lambda_
  u0 ← gamma*v0/beta - b13*alpha**2/lambda_
  K ← np.array([[alpha,gamma,u0],[0,beta,v0],[0,0,1]], dtype=float)

  # 9. RECOVER CAMERA POSES
  FUNCTION recover_camera_pose(K,H):
      K_inv ← np.linalg.inv(K)
      h1 ← H[:,0]; h2 ← H[:,1]; h3 ← H[:,2]
      lambda_p ← 1.0 / np.linalg.norm(K_inv @ h1)
      r1 ← lambda_p*(K_inv@h1)
      r2 ← lambda_p*(K_inv@h2)
      r3 ← np.cross(r1,r2)
      R_approx ← np.column_stack((r1,r2,r3))
      U,_,Vt ← np.linalg.svd(R_approx)
      R ← U @ Vt
      IF np.linalg.det(R) < 0:
          U[:,-1] ← -U[:,-1]
          R ← U @ Vt
      t ← lambda_p*(K_inv@h3)
      RETURN R,t
  FOR view IN views: view["R"],view["t"] ← recover_camera_pose(K,view["H"])

  # 10. REPROJECT POINTS
  FOR view IN views:
      chessboard_xy_points ← view["plane_pts"]
      chessboard_xyz_points ← np.column_stack((chessboard_xy_points, np.zeros(chessboard_xy_points.shape[0])))
      camera_frame_points ← chessboard_xyz_points @ view["R"].T + view["t"]
      projected_h ← camera_frame_points @ K.T
      IF np.any(np.abs(projected_h[:,2]) < 1e-12): RAISE ValueError
      view["world_xyz"] ← chessboard_xyz_points
      view["projected"] ← projected_h[:,:2] / projected_h[:,2:3]

  # 11. ERRORS AND RMSE
  FOR view IN views:
      view["residuals"] ← view["projected"] - view["im_pts"]
      view["errors"] ← np.linalg.norm(view["residuals"], axis=1)
      view["mean_error"] ← float(np.mean(view["errors"]))
      view["rmse"] ← float(np.sqrt(np.mean(view["errors"]**2)))
  all_reprojection_errors ← np.concatenate([view["errors"] for view in views])
  overall_mean_error ← float(np.mean(all_reprojection_errors))
  overall_rmse ← float(np.sqrt(np.mean(all_reprojection_errors**2)))

  # 12. SIX REQUIRED FIGURES
  # 12.1 Homography pipeline: figsize=(15,3.5), six boxed labels, arrows, dpi=300.
  # 12.2 Mean error by view: figsize=(10,5), bars, dashed overall-mean line, value labels, y-grid.
  # 12.3 Error distribution: figsize=(9,5), histogram bins=20, edgecolor=black, alpha=0.8, mean and RMSE lines.
  # 12.4 Detected corners: 3-column subplot grid, cv2.drawChessboardCorners with (INTERNAL_CORNERS_X, INTERNAL_CORNERS_Y), BGR→RGB, hide unused axes.
  # 12.5 Camera poses: 3D plot, board at Z=0, camera_center=-R.T@t, label each with path stem.
  # 12.6 Detected vs reprojected: 3-column grid, RGB image, detected marker=o, reprojected marker=x, title includes RMSE to 3 decimals.
  # For every figure: plt.tight_layout(); savefig(..., dpi=300, bbox_inches="tight"); plt.show().

  # 13. FINAL VALIDATION
  IF K.shape != (3,3) OR NOT np.all(np.isfinite(K)): RAISE ValueError
  FOR view IN views:
      IF NOT np.all(np.isfinite(view["H"])): RAISE ValueError
      IF NOT np.allclose(view["R"].T @ view["R"], np.eye(3), atol=1e-6): RAISE ValueError
      IF NOT np.isclose(np.linalg.det(view["R"]), 1.0, atol=1e-6): RAISE ValueError
      IF NOT np.all(np.isfinite(view["errors"])): RAISE ValueError
  missing ← [name for name in REQUIRED_OUTPUTS if not (OUTPUT_DIR/name).exists()]
  IF missing: RAISE FileNotFoundError("Missing outputs: " + ", ".join(missing))
  PRINT "All Camera Calibration validation checks passed."

END ALGORITHM
```
